# Modeling Part 3 - Experiments with the fly brain model

In Part 2, you rebuilt the model of Shiu et al. (2024) for the sugar → MN9 circuit and saw that stimulating the sugar GRNs activates MN9. In this notebook we use the model for experiments, as Shiu et al. did:
1. change the stimulation frequency and see how MN9 responds,
2. silence neurons to find out which ones the sugar signal needs to reach MN9,
3. (optional) run the same model on the whole fly brain, with all ~140,000 neurons.

These runs take longer than the single run in Part 2: from a few seconds for each experiment on our circuit, up to a few minutes for the whole brain.

Gaps for you to fill in are marked `___`.

## 0. Setup
As in parts 1 and 2, [click here](https://colab.research.google.com/github/ryanrahy/fcnc_workshop/blob/master/day2/modeling_part3_experiments.ipynb) to run this notebook on Colab. If you are running this notebook on Google Colab (or any fresh Python environment), run the next cell to install Brian 2.

In [ ]:
# get code + data
!git clone https://github.com/ryanrahy/fcnc_workshop.git
import os
os.chdir('./fcnc_workshop/day2/')

# install brian
!pip install brian2

In [ ]:
# import packages
from textwrap import dedent   # used in the parameter dictionary below
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from brian2 import *
%matplotlib inline

# our circuit is small, so plain NumPy is fast enough (see Section 4 for the whole brain)
prefs.codegen.target = 'numpy'

## 1. The model

The next cells repeat what you did in Part 2. They load the sugar → MN9 circuit (the two files saved at the end of FlyWire intro Part 3, which need to be in the same folder as this notebook) and the parameters of Shiu et al.

In [ ]:
# read in network connectivity info (see Part 2, Section 1)
W = pd.read_csv('../day1/sugar_circuit_signed_adjacency.csv', index_col=0)   # rows: presynaptic, columns: postsynaptic
W.columns = W.columns.astype('int64')   # the column names (root IDs) are read as text
neurons = pd.read_csv('../day1/sugar_circuit_neurons.csv').set_index('root_id').loc[W.index]   # same order as W

# Brian indices of the sugar GRNs (layer 0) and the two MN9s (layer 3)
sugar_idx = np.flatnonzero(neurons.layer == 0)
mn9_idx = np.flatnonzero(neurons.layer == 3)

print(f'{len(neurons)} neurons, {(W != 0).values.sum()} connections, '
      f'{len(sugar_idx)} sugar GRNs, {len(mn9_idx)} MN9 neurons')

In [ ]:
params = {
    # trials
    't_run'     : 1000 * ms,              # duration of trial
    'n_run'     : 30,                     # number of runs

    # network constants
    # Kakaria and de Bivort 2017 https://doi.org/10.3389/fnbeh.2017.00008
    'v_0'       : -52 * mV,               # resting potential
    'v_rst'     : -52 * mV,               # reset potential after spike
    'v_th'      : -45 * mV,               # threshold for spiking
    't_mbr'     :  20 * ms,               # membrane time scale (capacitance * resistance = .002 * uF * 10. * Mohm)

    # Jürgensen et al https://doi.org/10.1088/2634-4386/ac3ba6
    'tau'       : 5 * ms,                 # time constant 

    # Lazar et al https://doi.org/10.7554/eLife.62362
    't_rfc'     : 2.2 * ms,               # refractory period

    # Paul et al 2015 doi: 10.3389/fncel.2015.00029
    't_dly'     : 1.8*ms,                 # delay for changes in post-synaptic neuron

    # Free parameter 
    'w_syn'     : .275 * mV,              # weight per synapse (note: modulated by exponential decay)
    # Default activation rates 
    'r_poi'     : 150*Hz,                 # default rate of the Poisson inputs
    'r_poi2'    :   0*Hz,                 # default rate of a 2nd class of Poisson inputs
    'f_poi'     : 250,                    # scaling factor for Poisson synapse; 250 is sufficient to cause spiking

    # equations for neurons               # alpha synapse https://doi.org/10.1017/CBO9780511815706; See https://brian2.readthedocs.io/en/stable/user/converting_from_integrated_form.html
    'eqs'       : dedent(''' 
                    dv/dt = (v_0 - v + g) / t_mbr : volt (unless refractory)
                    dg/dt = -g / tau               : volt (unless refractory) 
                    rfc                            : second
                    '''),
    # condition for spike
    'eq_th'     : 'v > v_th', 
    # rules for spike        
    'eq_rst'    : 'v = v_rst; w = 0; g = 0 * mV', 
}

To run several experiments, we collect the model code from Part 2, Section 2 into a function. It takes the connections as lists (`pre`, `post` and the signed number of synapses), so that we can use it for the whole brain later. It returns the firing rate of every neuron. The extra argument `silenced` is new: it sets the weights of all outgoing synapses of these neurons to 0, as Shiu et al. did to model silencing a neuron (e.g. with an optogenetic inhibitor).

In [ ]:
def run_model(n_neurons, pre, post, n_synapses, stim_idx, params, silenced=()):
    start_scope()

    # neurons (Part 2, 2.1)
    neu = NeuronGroup(n_neurons, params['eqs'], method='linear',
                      threshold=params['eq_th'], reset=params['eq_rst'],
                      refractory='rfc', namespace=params)
    neu.v = params['v_0']
    neu.g = 0*mV
    neu.rfc = params['t_rfc']

    # synapses (Part 2, 2.2)
    syn = Synapses(neu, neu, model='w : volt', on_pre='g_post += w', delay=params['t_dly'])
    syn.connect(i=pre, j=post)
    syn.w = n_synapses * params['w_syn']
    for k in silenced:
        syn.w[f'i == {k}'] = 0*mV   # silence neuron k: remove all its outputs

    # stimulation (Part 2, 2.3)
    stim = PoissonGroup(len(stim_idx), rates=params['r_poi'])
    stim_syn = Synapses(stim, neu, model='w : volt', on_pre='v_post += w')
    stim_syn.connect(i=np.arange(len(stim_idx)), j=stim_idx)
    stim_syn.w = params['w_syn'] * params['f_poi']
    neu.rfc[stim_idx] = 0*ms

    # run (Part 2, 2.4)
    spike_mon = SpikeMonitor(neu)
    run(params['t_run'])
    return spike_mon.count / params['t_run'] / Hz


# the connections of our circuit, as lists
pre, post = np.nonzero(W.to_numpy())
n_synapses = W.to_numpy()[pre, post]

Before experimenting, we run the model once without any manipulation, exactly as in Part 2. We'll compare every experiment with this baseline. The table lists the most active neurons, apart from the stimulated GRNs.

In [ ]:
neurons['rate'] = run_model(len(W), pre, post, n_synapses, sugar_idx, params)

print(f'{(neurons.rate > 0).sum()} of {len(neurons)} neurons fired')
print('MN9 firing rates (Hz):', neurons.rate.iloc[mn9_idx].round(1).to_dict())
neurons[neurons.layer > 0].sort_values('rate', ascending=False).head(15)

**Check:** as in Part 2, both MN9s fire, one at roughly 80 Hz and the other at roughly 50 Hz.

## 2. Stimulation frequency
How does MN9 respond to weaker or stronger sugar stimulation? The loop below runs the model for several stimulation frequencies. `dict(params, r_poi=rate)` makes a copy of `params` with a different value for `r_poi`.

In [ ]:
stim_rates = [10, 25, 50, 100, 150, 200] * Hz
mn9_rates = []
for rate in stim_rates:
    firing_rates = run_model(len(W), pre, post, n_synapses, sugar_idx, dict(params, r_poi=rate))
    mn9_rates.append(firing_rates[mn9_idx])

plt.figure(figsize=(5, 3.5))
plt.plot(stim_rates/Hz, mn9_rates, 'o-', label=[f'MN9 (neuron {k})' for k in mn9_idx])
plt.xlabel('Sugar GRN stimulation (Hz)')
plt.ylabel('MN9 firing rate (Hz)')
plt.legend();

**Check:** MN9 stays silent up to 50 Hz of stimulation and only responds above that. Beyond this threshold, its firing rate grows roughly in proportion to the stimulation. Like the f–I curves in Part 1, the input–output relationship of the circuit is not linear.

## 3. Silencing neurons
Shiu et al. tested the model's predictions by **silencing** neurons, and then did the same experiment in real flies. Which neurons does the sugar signal need to reach MN9? Choose neurons to silence with a condition on the `neurons` table, for example:
* all layer-1 neurons: `neurons.layer == 1`
* all GABAergic neurons: `neurons.top_nt == 'gaba'`
* a named neuron from the baseline table in Section 1: `neurons.name.str.contains('roundup')`

**Predict** first: what will happen to MN9 in each case?

<details><summary>Show solution</summary>

Any condition on the `neurons` table works, for example:
```python
silenced = np.flatnonzero(neurons.layer == 1)                       # MN9 falls silent
silenced = np.flatnonzero(neurons.top_nt == 'gaba')                 # MN9 fires much faster
silenced = np.flatnonzero(neurons.name.str.contains('roundup'))     # MN9 drops sharply
```
</details>

In [ ]:
silenced = np.flatnonzero(___)   # Brian indices of the neurons to silence
print(len(silenced), 'neurons silenced:', neurons.name.iloc[silenced].unique()[:10])

firing_rates = run_model(len(W), pre, post, n_synapses, sugar_idx, params, silenced=silenced)
print('MN9 firing rates (Hz):', firing_rates[mn9_idx], ' (without silencing:', neurons.rate.iloc[mn9_idx].values, ')')

**Think:**
* Silencing all inhibitory (GABAergic) neurons *increases* MN9's firing. What does that tell you about the role of inhibition in this circuit?
* Silencing all 49 layer-1 neurons abolishes MN9's response. Can you find a single named cell type (usually 2 neurons, one per side) whose silencing has almost the same effect? Try the named neurons from the baseline table in Section 1. Shiu et al. used their model in the same way, to choose which neurons to silence in real flies.

<details><summary>Show answer</summary>

Silencing `roundup` (MN9 drops to roughly 3 and 25 Hz) or `G2N_1` (roughly 0 and 10 Hz) almost abolishes MN9's response. Silencing most other neurons, e.g. `rattle`, `FMIn` or `clavicle`, changes MN9 by less than 10%.
</details>

## 4. (Optional) The whole brain
Our circuit only contains the neurons on short paths from the sugar GRNs to MN9, and only connections with at least 5 synapses. The real model of Shiu et al. contains *every* neuron and *every* connection in FlyWire: about 140,000 neurons and 15 million connections. Thanks to `run_model`, we can simulate it with exactly the same code. Only the connectivity changes.

The connectivity comes from the authors' repository as two files (about 100 MB, downloaded by the next cell):
* `Completeness_783.csv`: a list of all neurons. Row `k` is neuron `k` of the model.
* `Connectivity_783.parquet`: a table with one row per connection, with the Brian indices of the presynaptic and postsynaptic neuron (`Presynaptic_Index`, `Postsynaptic_Index`) and the signed number of synapses (`Excitatory x Connectivity`). This is the same information as our `W`, as a list instead of a matrix.

For a network of this size, we switch Brian back to generating C++ code. Expect the first run to take one to a few minutes, longer on Colab.

In [ ]:
import os
import urllib.request

url = 'https://github.com/philshiu/Drosophila_brain_model/raw/main/'
for file in ['Completeness_783.csv', 'Connectivity_783.parquet']:
    if not os.path.exists(file):
        urllib.request.urlretrieve(url + file, file)

completeness = pd.read_csv('Completeness_783.csv', index_col=0)
connectivity = pd.read_parquet('Connectivity_783.parquet')
print(len(completeness), 'neurons,', len(connectivity), 'connections')
connectivity.head()

In [ ]:
# Brian index of every root ID in the whole-brain model
brain_idx = {root_id: k for k, root_id in enumerate(completeness.index)}
sugar_brain = [brain_idx[r] for r in neurons.index[sugar_idx]]
mn9_brain = [brain_idx[r] for r in neurons.index[mn9_idx]]

prefs.codegen.target = 'cython'   # generate C++ code: much faster for large networks
rates_brain = run_model(len(completeness),
                        connectivity['Presynaptic_Index'].values,
                        connectivity['Postsynaptic_Index'].values,
                        connectivity['Excitatory x Connectivity'].values,
                        sugar_brain, params)

print(f'{(rates_brain > 0).sum()} neurons fired')
print('MN9 firing rates (Hz), whole brain:', rates_brain[mn9_brain], ' circuit:', neurons.rate.iloc[mn9_idx].values)

# how many of the active neurons are in our circuit?
active = completeness.index[rates_brain > 0]
print(f'{np.isin(active, neurons.index).sum()} of the {len(active)} active neurons are in our circuit')

**Check:** nearly 400 neurons fire in the whole brain, but only about 85 of them are in our circuit. MN9 responds about as strongly as in our circuit. (A single 1 s run is noisy: averaged over several runs, one MN9 fires at roughly 80 Hz in both models, and the other at roughly 50 Hz in the circuit and 60 Hz in the whole brain.)

**Think:**
* Our circuit leaves out more than 99.8% of the neurons and every connection with fewer than 5 synapses, yet it reproduces MN9's response to sugar almost exactly. What does that tell you about the sugar → MN9 pathway? Would you expect this for every input and output pair?
* Most of the active neurons in the whole brain are not in our circuit. Where could they be, and what could they be doing?
* What would you need to change to model bitter taste instead (Part 3, Exercise 1)?

### References
* Shiu et al. (2024) *A Drosophila computational brain model reveals sensorimotor processing.* Nature 634, 210–219. Code: [github.com/philshiu/Drosophila_brain_model](https://github.com/philshiu/Drosophila_brain_model)
* Stimberg, Brette & Goodman (2019) *Brian 2, an intuitive and efficient neural simulator.* eLife 8, e47314.